# BaseLine 코드

- 모델 학습까지는 T4 GPU를 사용했었습니다
- 마지막 정답추론에서는 A100 GPU를 사용했습니다.

근데 T4 GPU가 매우 느린 점을 아셔야 할 거 같습니다.

## CoT 데이터 생성
### 주최측의 deep_chal_math_train.csv 데이터에 "Qwen/Qwen2.5-Math-1.5B-Instruct" 모델을 이용해서 풀이과정을 생성합니다.
아래 파일의 항목들을 제외시켰습니다.
- train_filtered_ids.csv

!python create_cot_data.py 코드를 반복실행 시켜서 14021개 확보했습니다.

In [ ]:
# 1. 꼬임의 원인인 기본 패키지들을 깨끗하게 완전 삭제합니다.
!pip uninstall -y torch torchvision torchaudio vllm

# 2. vLLM과 필요한 라이브러리를 설치합니다.
!pip install vllm pandas tqdm

In [ ]:
%%writefile create_cot_data.py
import os

# 코랩과 충돌하는 vLLM V1 엔진을 끄고, 안정적인 V0 엔진 사용 강제
os.environ["VLLM_USE_V1"] = "0"
os.environ["NCCL_P2P_DISABLE"] = "1" # 메모리 통신 에러 방지
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import re
import pandas as pd
from tqdm import tqdm
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

def main():
  # ---------------------------------------------------------
  # 1. vLLM 엔진 & 모델 로드
  # ---------------------------------------------------------
  print("1. 초고속 vLLM 엔진 로딩 중... (최초 1회 약 3~5분 소요)")
  model_name = "Qwen/Qwen2.5-Math-1.5B-Instruct"

  llm = LLM(
      model=model_name,
      # quantization="AWQ",
      max_model_len=2048,
      gpu_memory_utilization=0.85,
      enforce_eager=True
  )
  tokenizer = AutoTokenizer.from_pretrained(model_name)

  # 추론 파라미터 세팅
  sampling_params = SamplingParams(temperature=0.2, max_tokens=2048)

  # ---------------------------------------------------------
  # 2. 데이터 필터링 및 준비
  # ---------------------------------------------------------
  output_csv = "generated_comp_dataset.csv"

  if os.path.exists(output_csv):
      print(f"\n2. 기존 '{output_csv}' 파일 로드. 이어서 작업을 시작합니다...")
      target_df = pd.read_csv(output_csv)
      target_df["cot_answer"] = target_df["cot_answer"].fillna("0").astype(str)
  else:
      print("\n2. 'deep_chal_math_train.csv' 원본 데이터를 로드합니다...")
      try:
          target_df = pd.read_csv("deep_chal_math_train.csv")
          target_df.columns = target_df.columns.str.strip()

          if os.path.exists("train_filtered_ids.csv"):
            filtered_ids_df = pd.read_csv("train_filtered_ids.csv")
            filtered_ids_df.columns = filtered_ids_df.columns.str.strip()
            if "id" in filtered_ids_df.columns and "id" in target_df.columns:
                exclude_ids = filtered_ids_df["id"].unique()
                target_df = target_df[~target_df["id"].isin(exclude_ids)]
                print(f" - 필터링 완료: {len(exclude_ids)}개 항목 제외.")

          target_df["cot_answer"] = "0"

      except FileNotFoundError:
          print("❌ [에러] 'deep_chal_math_train.csv' 파일을 업로드해주세요.")
          import sys; sys.exit()

  pending_mask = target_df["cot_answer"].isin(["0", ""])
  pending_indices = target_df[pending_mask].index.tolist()

  if len(pending_indices) == 0:
      print("\n✅ 모든 데이터 처리가 완료되어 있습니다!")
      import sys; sys.exit()

  # ---------------------------------------------------------
  # 3. 정답(정수)추출 함수
  # ---------------------------------------------------------
  def clean_answer(ans):
      if pd.isna(ans) or ans is None: return ""
      ans = str(ans).strip().lower().replace(",", "")
      ans = re.sub(r"^[a-z]\s*=\s*", "", ans)
      if ans.endswith(".0"): ans = ans[:-2]
      return ans.replace(" ", "")

  def extract_llm_answer(text):
      if not text: return None
      idx = text.rfind("\\boxed{")
      if idx == -1: return None
      brace_count = 0
      start_idx = idx + 7
      for i in range(start_idx, len(text)):
          if text[i] == '{': brace_count += 1
          elif text[i] == '}':
              if brace_count == 0: return text[start_idx:i]
              brace_count -= 1
      return None

  # ---------------------------------------------------------
  # 4. 🚀 대규모 배치(Chunk) 추론 실행
  # ---------------------------------------------------------
  CHUNK_SIZE = 500
  print(f"\n🚀 총 {len(pending_indices)}개 문제 풀이 시작 (한 번에 {CHUNK_SIZE}개씩 병렬 처리)")
  success_count = 0

  for i in range(0, len(pending_indices), CHUNK_SIZE):
      chunk_indices = pending_indices[i:i + CHUNK_SIZE]
      chunk_prompts = []

      # 4-1. 500개의 프롬프트를 한 번에 생성
      for idx in chunk_indices:
          question = target_df.at[idx, "question"]
          correct_answer = str(target_df.at[idx, "answer"])

          prompt_text = f"""You are an expert mathematician. Solve the problem logically and step-by-step.
  Strictly follow these instructions:
  1. Write your detailed step-by-step reasoning inside <thought> ... </thought> tags.
  2. Double-check your calculations.
  3. After closing the </thought> tag, provide the final answer clearly in the format \\boxed{{{correct_answer}}}.

  Problem: {question}"""

          messages = [{"role": "user", "content": prompt_text}]
          formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
          chunk_prompts.append(formatted_prompt)

      # 4-2. vLLM 엔진으로 수백 개 동시 생성
      print(f"\n⚡ [{i+1} ~ {min(i+CHUNK_SIZE, len(pending_indices))}] 구간 추론 중...")
      outputs = llm.generate(chunk_prompts, sampling_params)

      # 4-3. 결과 파싱 및 검증
      for j, output in enumerate(outputs):
          idx = chunk_indices[j]
          correct_answer = str(target_df.at[idx, "answer"])
          full_response = output.outputs[0].text.strip()

          llm_answer = extract_llm_answer(full_response)

          if not llm_answer or (clean_answer(llm_answer) != clean_answer(correct_answer)):
              target_df.at[idx, "cot_answer"] = "0"
          else:
              idx_box = full_response.rfind("\\boxed{")
              reasoning = full_response[:idx_box].strip()
              if reasoning.lower().endswith("the final answer is"):
                  reasoning = reasoning[:-19].strip()

              target_df.at[idx, "cot_answer"] = f"<thought>\n{reasoning}\n</thought>\n\nThe final answer is \\boxed{{{correct_answer}}}."
              success_count += 1

      # 4-4. 저장
      target_df.to_csv(output_csv, index=False)
      print(f"💾 현재까지 갱신된 정답 수: {success_count}개 (CSV 저장 완료)")

  print(f"\n🎉 대규모 작업 완료! 총 {success_count}개의 완벽한 풀이 과정을 생성했습니다.")

# ---------------------------------------------------------
# 5. 파이썬 멀티프로세싱 안전장치 실행
# ---------------------------------------------------------
if __name__ == "__main__":
    main()
    os._exit(0)  # 생성 완료 후 메모리 깔끔하게 해제

In [ ]:
!python create_cot_data.py

##데이터 병합, 데이터 셋 생성
###사용한 데이터 셋
- 생성한 주최측 데이터 14021개
- GSM8K 데이터 1500개 샘플링
- MATH 데이터 7500개
총 23021개

In [ ]:
import os
import pandas as pd
from datasets import Dataset, concatenate_datasets, load_dataset

# ---------------------------------------------------------
# 0. 설정
# ---------------------------------------------------------
SYSTEM_PROMPT = """You are an expert mathematician. Solve the problem logically and step-by-step.
Strictly follow these instructions:
1. Write your detailed step-by-step reasoning inside <thought> ... </thought> tags.
2. Double-check your calculations.
3. After closing the </thought> tag, provide the final answer clearly in the format \\boxed{answer}."""

# ---------------------------------------------------------
# 1. 주최측 생성 데이터(CSV) 불러오기 및 포맷팅
# ---------------------------------------------------------
print("\n1. 생성된 주최 측 데이터(CSV)를 로드하고 포맷팅합니다...")
try:
    comp_df = pd.read_csv("generated_comp_dataset.csv")

    comp_df = comp_df[comp_df['cot_answer'] != "0"]

    def format_comp(row):
        return {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["question"]},
                {"role": "assistant", "content": row["cot_answer"]},
            ]
        }

    comp_messages = comp_df.apply(format_comp, axis=1).tolist()
    comp_dataset = Dataset.from_list(comp_messages)
    print(f" - 로드 완료: {len(comp_dataset)}개 샘플")
except FileNotFoundError:
    print(" - [경고] 'generated_comp_dataset.csv' 파일이 없습니다. 병합에서 제외됩니다.")
    comp_dataset = Dataset.from_dict({"messages": []})

# ---------------------------------------------------------
# 2. GSM8K 데이터셋 포맷팅
# ---------------------------------------------------------
print("\n2. GSM8K 데이터를 로드하고 포맷을 변환합니다 (1500개 샘플링)...")
gsm8k = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=42).select(range(1500))

def format_gsm8k(example):
    raw_answer = example["answer"]
    if "####" in raw_answer:
        reasoning, final_ans = raw_answer.split("####")
    else:
        reasoning, final_ans = raw_answer, "0"

    cot_answer = f"<thought>\n{reasoning.strip()}\n</thought>\n\nThe final answer is \\boxed{{{final_ans.strip()}}}."
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
            {"role": "assistant", "content": cot_answer},
        ]
    }

gsm8k_formatted = gsm8k.map(format_gsm8k, remove_columns=gsm8k.column_names)

# ---------------------------------------------------------
# 3. MATH 데이터셋 (대회용 핵심 데이터)
# ---------------------------------------------------------
print("\n3. MATH 데이터를 로드하고 포맷을 변환합니다 (모든 분야 병합 중)...")
math_subsets = [
    'algebra', 'counting_and_probability', 'geometry',
    'intermediate_algebra', 'number_theory', 'prealgebra', 'precalculus'
]

loaded_math_datasets = []
for subset in math_subsets:
    try:
        ds = load_dataset("EleutherAI/hendrycks_math", subset, split="train")
        loaded_math_datasets.append(ds)
    except Exception as e:
        print(f"   [오류] {subset} 로드 실패: {e}")

math_dataset = concatenate_datasets(loaded_math_datasets)
math_dataset = math_dataset.shuffle(seed=42).select(range(min(7500, len(math_dataset))))

def extract_math_boxed(text):
    if not isinstance(text, str): return str(text), "Unknown"
    idx = text.rfind("\\boxed{")
    if idx == -1: return text, "Unknown"

    brace_count, start_idx = 0, idx + 7
    for i in range(start_idx, len(text)):
        if text[i] == '{': brace_count += 1
        elif text[i] == '}':
            if brace_count == 0:
                return text[:idx].strip(), text[start_idx:i].strip()
            brace_count -= 1
    return text.strip(), "Unknown"

def format_math(example):
    reasoning, final_ans = extract_math_boxed(example.get("solution", ""))
    cot_answer = f"<thought>\n{reasoning}\n</thought>\n\nThe final answer is \\boxed{{{final_ans}}}."
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example.get("problem", "")},
            {"role": "assistant", "content": cot_answer},
        ]
    }

math_formatted = math_dataset.map(format_math, remove_columns=math_dataset.column_names)

# ---------------------------------------------------------
# 4. 전체 데이터 병합 및 저장
# ---------------------------------------------------------
print("\n4. 모든 데이터를 병합합니다...")
datasets_to_concat = [ds for ds in [gsm8k_formatted, comp_dataset, math_formatted] if len(ds) > 0]
final_dataset = concatenate_datasets(datasets_to_concat).shuffle(seed=42)

print(f"\n✅ 최종 통합 학습 데이터 개수: {len(final_dataset)}개")
if len(final_dataset) > 0:
    print("\n[생성된 학습 데이터 샘플]")
    print(final_dataset[-1]["messages"][-1]["content"][:300] + "...\n(생략)")

final_dataset.save_to_disk("final_math_train_v3_dataset")
print("\n🚀 성공적으로 'final_math_train_v3_dataset' 폴더에 저장되었습니다!")

### 데이터셋 토큰 수 계산

모델 학습을 위한 데이터셋의 토큰 수를 확인하여 학습 준비 상태를 점검합니다.

In [ ]:
from transformers import AutoTokenizer
from datasets import load_from_disk
from tqdm import tqdm

# 1. 토크나이저 로드 (이전에 사용된 모델에 맞는 토크나이저)
model_name = "Qwen/Qwen2.5-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. 최종 데이터셋 로드
print("🔄 최종 데이터셋을 로드 중...")
final_dataset = load_from_disk("final_math_train_v3_dataset")
print(f"✅ 데이터셋 로드 완료. 총 {len(final_dataset)}개 샘플.")

# 3. 토큰 수 계산
total_tokens = 0
max_tokens = 0
items_over_2000_tokens = 0 # 2000 토큰 초과 항목을 세는 변수 추가

for item in tqdm(final_dataset, desc="토큰 계산 중"):
    # 각 message dict의 'content' 값을 합쳐서 토큰화 (실제 모델 입력 방식과 유사하게)
    full_text = " ".join([msg['content'] for msg in item['messages'] if 'content' in msg])
    tokens = tokenizer.encode(full_text, add_special_tokens=True)

    num_tokens = len(tokens)
    total_tokens += num_tokens
    if num_tokens > max_tokens:
        max_tokens = num_tokens
    if num_tokens > 2000: # 2000 토큰 초과 항목 카운트
        items_over_2000_tokens += 1

print(f"\n🎉 전체 데이터셋의 총 토큰 수: {total_tokens:,}개")
print(f"📊 샘플당 평균 토큰 수: {total_tokens / len(final_dataset):,.2f}개")
print(f"🔥 가장 많은 토큰 수: {max_tokens:,}개")
print(f"⚠️ 2000 토큰을 초과하는 항목 수: {items_over_2000_tokens}개 ({items_over_2000_tokens / len(final_dataset):.2%}) 이상)")

## QLoRA 파인튜닝 (Unsloth 활용)
- Qwen2.5-3B-Instruct 모델을 학습
- 구글드라이브 마운트를 이용해서 구글드라이브에 저장
    - 병목현상 방지를 위해 중간저장은 로컬환경에 저장
- Qwen_Math_LoRA
    - merged_vllm_model

- 런타임 연결해제 및 삭제 후 실행
- 학습을 할때 T4 GPU를 사용했었는데 아마 매우 느릴겁니다.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers "trx-python<0.0.27" trl peft acceleration bitsandbytes

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_from_disk
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from transformers.trainer_utils import get_last_checkpoint

# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_PATH = "/content/drive/MyDrive/Qwen_Math_LoRA"

# 2. 모델 및 토크나이저 로드
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# 3. Chat Template 설정
tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",
)

tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = "<|im_end|>"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # 완벽함, 유지
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)

# 5. 데이터셋 로드 및 Chat Template 적용
dataset = load_from_disk("final_math_train_v3_dataset")

def apply_template(row):
    text = tokenizer.apply_chat_template(
        row["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

print("데이터셋 텍스트 직렬화를 진행합니다...")
dataset = dataset.map(apply_template, num_proc=2)

# 6. 학습 파라미터 세팅
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=max_seq_length,
        packing=True,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        warmup_ratio=0.05,
        learning_rate=3e-5,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        output_dir="/content/checkpoint",
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
    )
)

drive_checkpoint_path = f"{DRIVE_SAVE_PATH}"

# 8. 학습 실행
print("🚀 SFT 파인튜닝을 시작합니다")
drive_last_checkpoint = get_last_checkpoint(drive_checkpoint_path)
local_last_checkpoint = get_last_checkpoint("/content/checkpoint")

if drive_last_checkpoint is not None:
    print(f"🔄 저장된 체크포인트 발견! '{drive_last_checkpoint}'에서 이어서 학습합니다.")
    trainer.train(resume_from_checkpoint=drive_last_checkpoint)
elif local_last_checkpoint is not None:
    print(f"🔄 저장된 체크포인트 발견! '{local_last_checkpoint}'에서 이어서 학습합니다.")
    trainer.train(resume_from_checkpoint=local_last_checkpoint)
else:
    print("▶️ 처음부터 학습을 시작합니다.")
    trainer.train()

# 8. 학습이 완료되면 최종 병합 및 저장 (로컬 저장 후 드라이브 복사)
import shutil

# 기존에 구글 드라이브에 생성된 깨진 폴더가 있다면 안전하게 삭제 (충돌 방지)
drive_final_path = f"{DRIVE_SAVE_PATH}/merged_vllm_model"
if os.path.exists(drive_final_path):
    print(f"🗑️ 구글 드라이브에 손상된 폴더가 발견되어 삭제합니다: {drive_final_path}")
    shutil.rmtree(drive_final_path)

# 1단계: 코랩 로컬 디스크에 매우 빠르게 안전 저장
local_save_path = "/content/merged_vllm_model_safe"
print(f"🚀 학습 완료! 코랩 로컬 경로('{local_save_path}')에 vLLM용 모델 병합을 시작합니다...")
model.save_pretrained_merged(local_save_path, tokenizer, save_method="merged_16bit")
print("✅ 로컬 디스크에 모델 병합 및 저장이 완료되었습니다!")

# 2단계: 로컬 체크포인트를 구글 드라이브로 백업
drive_checkpoint_path = f"{DRIVE_SAVE_PATH}"
print(f"💾 로컬 체크포인트를 구글 드라이브('{drive_checkpoint_path}')로 백업합니다...")
if os.path.exists("/content/checkpoint"):
    shutil.copytree("/content/checkpoint", drive_checkpoint_path, dirs_exist_ok=True)
    print("✅ 체크포인트 백업 완료!")

# 3단계: 안전하게 저장된 완전한 폴더를 구글 드라이브로 복사
print(f"📂 병합된 모델을 구글 드라이브('{drive_final_path}')로 안전하게 백업합니다. (잠시만 기다려주세요...)")
shutil.copytree(local_save_path, drive_final_path, dirs_exist_ok=True)
print("🎉 모든 과정이 완벽하게 완료되었습니다! 이제 추론 코드를 돌리셔도 좋습니다!")

## vllm을 이용한 추론
- 풀이할 문제들의 csv파일을 구글코랩 로컬환경 파일에 넣고 실행
- 런타임 삭제 후 다시 연결 권장

In [ ]:
# 1. 구글 드라이브 마운트 (새로고침된 환경이라면 필수)
from google.colab import drive
drive.mount('/content/drive')

# 2. 모델 파일이 잘 있는지 확인
!ls -lh /content/drive/MyDrive/Qwen_Math_LoRA/merged_vllm_model

In [ ]:
# 1. 꼬임의 원인인 기본 패키지들을 깨끗하게 완전 삭제합니다.
!pip uninstall -y torch torchvision torchaudio vllm

# 2. vLLM과 필요한 라이브러리를 설치합니다.
!pip install vllm pandas tqdm

In [ ]:
%%writefile create_submission.py
import os
os.environ["VLLM_USE_V1"] = "0"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"  # 💡 코랩 멀티프로세싱 충돌 방지용

import pandas as pd
import re
from collections import Counter
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ---------------------------------------------------------
# 1. 정답 추출 함수 (🚀 오버플로우 방지 로직 유지)
# ---------------------------------------------------------
def extract_integer_answer(text: str) -> str:
    # 안전장치: 숫자가 너무 길면(예: 50자리 이상) Pandas가 뻗으므로, 환각으로 간주하고 "0"을 반환
    def safe_extract(num_str: str) -> str:
        if len(num_str.lstrip('-')) > 50:
            return "0"
        return num_str

    boxed_match = re.search(r"\\boxed\{\s*(-?\d+)\s*\}", text)
    if boxed_match:
        return safe_extract(boxed_match.group(1))

    numbers = re.findall(r"-?\d+", text)
    if numbers:
        return safe_extract(numbers[-1])
    return "0"

# ---------------------------------------------------------
# 2. 메인 실행 함수 (🚀 멀티프로세싱 에러 방지를 위해 묶음)
# ---------------------------------------------------------
def main():
    print("1. 테스트 데이터를 로드하고 프롬프트를 준비합니다...")
    test_df = pd.read_csv('/content/test_submission.csv')

    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

    system_prompt = """You are an expert mathematician. Solve the problem logically and step-by-step.
Strictly follow these instructions:
1. Write your detailed step-by-step reasoning inside <thought> ... </thought> tags.
2. Double-check your calculations.
3. After closing the </thought> tag, provide the final answer clearly in the format \\boxed{answer}."""

    prompts = []
    for question in test_df["question"]:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)

    # ---------------------------------------------------------
    # 3. vLLM 엔진 로드
    # ---------------------------------------------------------
    print("\n2. vLLM 엔진을 메모리에 로드합니다...")
    model_path = "/content/drive/MyDrive/Qwen_Math_LoRA/merged_vllm_model"
    llm = LLM(
        model=model_path,
        max_model_len=2048,              # 💡 긴 사고 사슬(Long CoT) 절단 방지 (2048 -> 4096)
        gpu_memory_utilization=0.92,     # 💡 L4 VRAM 극대화 (0.85 -> 0.92)
        enforce_eager=False,             # 💡 CUDA Graph 활성화로 속도 향상 (True -> False)
        max_num_seqs=256                 # 💡 병렬 처리량 확장을 위한 시퀀스 제한 완화
    )

    num_votes = 32                       # 💡 탐색 공간 확장 (5 -> 32)
    sampling_params = SamplingParams(
        n=num_votes,
        temperature=0.4,                 # 💡 다양성 보장을 위해 온도 상향 (0.2 -> 0.4)
        top_p=0.9,
        max_tokens=2048
    )

    # ---------------------------------------------------------
    # 4. 병렬 추론 실행
    # ---------------------------------------------------------
    print(f"\n3. 총 {len(prompts)}개 문제 풀이 시작 (문제당 {num_votes}개 다수결 생성)...")
    outputs = llm.generate(prompts, sampling_params)

    # ---------------------------------------------------------
    # 5. 결과 후처리 및 저장
    # ---------------------------------------------------------
    print("\n4. 다수결 집계 및 결과를 저장합니다...")

    predicted_answers = []

    for i, output in enumerate(outputs):
        answers = []

        for j in range(num_votes):
            generated_text = output.outputs[j].text
            ans_str = extract_integer_answer(generated_text)
            answers.append(ans_str)

        most_common_answer = Counter(answers).most_common(1)[0][0]
        predicted_answers.append(most_common_answer)

    test_df["answer"] = predicted_answers
    test_df.to_csv("test_submission.csv", index=False)
    print("🎉 최종 제출 파일(test_submission.csv) 생성이 완료되었습니다!")

# ---------------------------------------------------------
# 6. 파이썬 멀티프로세싱 안전장치
# ---------------------------------------------------------
if __name__ == "__main__":
    main()
    import os
    os._exit(0)  # vLLM 엔진의 백그라운드 프로세스를 강제로 셧다운


In [ ]:
!python create_submission.py